In [20]:
import os
from google.colab import userdata

# Retrieve your Groq API Key from Colab's left sidebar Secrets menu (🔑 icon) named "GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = userdata.get('New_Api')

In [21]:
pip install langchain-groq langgraph -q

In [39]:
import os
from typing import Annotated, Literal
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver

# ========================================================
# 1. Define the Persistent Memory State
# ========================================================
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    step: str = "income"          # Stages: income -> expenses -> clarifying -> review_recommendation -> done
    income: float = 0.0
    expenses: float = 0.0
    risk_appetite: str = "Low"

# 2. Define Pydantic Schema for Structured Modification Extraction
class FinancialAdjustments(BaseModel):
    reasoning: str = Field(description="Brief explanation of what the user wants to change or why they dislike the plan.")
    income_update: float = Field(description="The new monthly income value if specified. If not mentioned or should remain unchanged, return the current income value exactly.")
    expenses_update: float = Field(description="The new monthly expenses value if specified. If not mentioned or should remain unchanged, return the current expenses value exactly.")
    risk_update: Literal["High", "Low", "Toggle"] = Field(description="Select 'High' or 'Low' if specified. If they ask for 'other recommendation' or 'something else' without details, select 'Toggle' to flip the current setting. If they don't want to change risk, match their current setting.")

# Initialize Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

# ========================================================
# 3. Node Logic with Fixed Structured Re-evaluation
# ========================================================
def finance_agent_node(state: AgentState):
    user_input = state.messages[-1].content if state.messages else ""

    # Track existing state parameters
    updated_step = state.step
    updated_income = state.income
    updated_expenses = state.expenses
    updated_risk = state.risk_appetite

    response = "I'm sorry, I couldn't process that step properly. Let's try again."

    # --- STAGE 1: Extract Income ---
    if updated_step == "income":
        res = llm.invoke(f"Extract only the total monthly income number from: '{user_input}'. Return just numbers.").content
        try:
            updated_income = float(''.join(c for c in res if c.isdigit() or c=='.'))
            updated_step = "expenses"
            response = "Got it! Now, what are your approximate monthly expenses? (e.g., rent, bills, subscriptions, coffee, travel, etc.)"
        except ValueError:
            response = "I couldn't quite catch the amount. Could you please state your monthly income clearly in digits?"

    # --- STAGE 2: Extract Expenses ---
    elif updated_step == "expenses":
        res = llm.invoke(f"Extract only the total monthly expense number from: '{user_input}'. Return just numbers.").content
        try:
            updated_expenses = float(''.join(c for c in res if c.isdigit() or c=='.'))
            updated_step = "clarifying"
            response = "Thanks. To customize your plan, how do you feel about investment risk? Would you prefer Low risk (safe, stable) or High risk (higher growth, volatile)?"
        except ValueError:
            response = "Could you please give me a rough estimate of your total monthly expenses as a number?"

    # --- STAGE 3: Extract Risk & Present Initial Recommendation ---
    elif updated_step == "clarifying":
        res = llm.invoke(f"Is this 'Low' or 'High' risk appetite?: '{user_input}'. Reply with just one word.").content.strip()
        updated_risk = "High" if "High" in res else "Low"
        updated_step = "review_recommendation"

        response = generate_financial_plan(updated_income, updated_expenses, updated_risk)
        response += "\n\nDoes this plan look good to you, or would you like to make any adjustments?"

    # --- STAGE 4: Confirmation Loop ---
    elif updated_step == "review_recommendation":
        # Step A: Validate Confirmation
        confirmation_prompt = (
            f"Analyze if the user is satisfied and confirming the plan, or if they want changes/dislike it. "
            f"User input: '{user_input}'. "
            f"Reply with exactly 'CONFIRMED' if they say yes, look good, agree, or thank you. "
            f"Otherwise, reply with 'CHANGES_REQUESTED'."
        )
        validation = llm.invoke(confirmation_prompt).content.strip()

        if "CONFIRMED" in validation:
            updated_step = "done"
            response = "Wonderful! I'm glad you found the plan helpful. Feel free to come back if your financial situation changes. Happy investing! 🚀"
        else:
            # Step B: Secure structured adjustments using structured outputs
            structured_llm = llm.with_structured_output(FinancialAdjustments)

            extraction_context = (
                f"The user is dissatisfied with their current plan. "
                f"Current State -> Income: {updated_income}, Expenses: {updated_expenses}, Risk: {updated_risk}.\n"
                f"User feedback input: '{user_input}'.\n"
                f"Extract their new desired profile. If they just say 'give me another recommendation', select 'Toggle' for the risk update."
            )

            try:
                result: FinancialAdjustments = structured_llm.invoke(extraction_context)

                # Apply updates from structured data object directly
                updated_income = result.income_update
                updated_expenses = result.expenses_update

                if result.risk_update == "Toggle":
                    updated_risk = "Low" if updated_risk == "High" else "High"
                elif result.risk_update in ["High", "Low"]:
                    updated_risk = result.risk_update
            except Exception:
                # Fallback switch if structure invocation fails to bind cleanly
                if "other" in user_input.lower() or "another" in user_input.lower() or "change" in user_input.lower():
                    updated_risk = "Low" if updated_risk == "High" else "High"

            # Step C: Generate clear conversational validation text response
            context_prompt = (
                f"You are a personal finance assistant. The user wanted to change or modify their strategy. "
                f"You have adjusted their profile to -> Income: {updated_income}, Expenses: {updated_expenses}, Risk Appetite: {updated_risk}.\n"
                f"User input was: '{user_input}'.\n"
                f"Write a 1-sentence friendly confirmation saying you have regenerated their strategy based on their request."
            )
            conversational_ack = llm.invoke(context_prompt).content

            # Step D: Always rebuild using clean structural logic
            new_plan = generate_financial_plan(updated_income, updated_expenses, updated_risk)
            response = f"{conversational_ack}\n\n{new_plan}\n\nDoes this updated setup work better for you, or would you like to make more changes?"

    return {
        "messages": [AIMessage(content=response)],
        "step": updated_step,
        "income": updated_income,
        "expenses": updated_expenses,
        "risk_appetite": updated_risk
    }

def generate_financial_plan(income: float, expenses: float, risk: str) -> str:
    surplus = income - expenses
    if surplus <= 0:
        return "⚠️ Your spending matches or exceeds your income. Prioritize setting up a 3-month basic **Emergency Fund** before investing."

    if risk == "Low":
        return f"💰 Monthly Surplus: ${surplus:.2f}.\nSince you prefer **Low Risk**, invest:\n- 70% in Fixed Deposits / Liquid Mutual Funds (Safe & Stable)\n- 30% in Debt Mutual Funds / Emergency Fund."
    else:
        return f"🚀 Monthly Surplus: ${surplus:.2f}.\nSince you prefer **High Risk**, invest:\n- 70% in Equity Mutual Funds via an **SIP**\n- 30% in Index Funds for long-term compounding growth."

# ========================================================
# 4. Compile State Graph Machine
# ========================================================
workflow = StateGraph(AgentState)
workflow.add_node("finance_agent", finance_agent_node)
workflow.add_edge(START, "finance_agent")
workflow.add_edge("finance_agent", END)

memory_storage = MemorySaver()
app = workflow.compile(checkpointer=memory_storage)

# ========================================================
# 5. Interactive Testing Loop
# ========================================================
config = {"configurable": {"thread_id": "user_session_loop_demo"}}

print("Agent: Hello! Let's map out your investments. What is your total monthly income?")

while True:
    user_msg = input("\nYou: ")
    if user_msg.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    output = app.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config=config
    )

    print(f"\nAgent: {output['messages'][-1].content}")

    if output["step"] == "done":
        print("\nSession complete! 🏁")
        break

Agent: Hello! Let's map out your investments. What is your total monthly income?

You: 20000

Agent: Got it! Now, what are your approximate monthly expenses? (e.g., rent, bills, subscriptions, coffee, travel, etc.)

You: 10000

Agent: Thanks. To customize your plan, how do you feel about investment risk? Would you prefer Low risk (safe, stable) or High risk (higher growth, volatile)?

You: high risk

Agent: 🚀 Monthly Surplus: $10000.00.
Since you prefer **High Risk**, invest:
- 70% in Equity Mutual Funds via an **SIP**
- 30% in Index Funds for long-term compounding growth.

Does this plan look good to you, or would you like to make any adjustments?

You: didint like it 

Agent: I've regenerated your personal finance strategy based on your updated profile and preferences, taking into account your income of $20,000, expenses of $10,000, and low risk appetite, to better suit your needs.

💰 Monthly Surplus: $10000.00.
Since you prefer **Low Risk**, invest:
- 70% in Fixed Deposits / Liquid 